# 04 - Deep Agent com sistema de arquivos

Nos notebooks anteriores, o agente planejou e chamou ferramentas, mas a resposta ficou só na conversa. Aqui ele também usa o **sistema de arquivos** do deep agent para guardar o resultado.

O `create_deep_agent` já traz ferramentas de arquivo, sem configuração extra:

- `write_file`: cria um arquivo;
- `read_file`: lê um arquivo;
- `edit_file`: altera um trecho de um arquivo;
- `ls`, `glob` e `grep`: lista e procura arquivos.

Neste notebook, o agente:

1. cria um plano com `write_todos`;
2. consulta as vendas com `consultar_vendas` e faz as contas com `calcular`;
3. grava o relatório em `/relatorio.md` com `write_file`;
4. lê o arquivo de volta com `read_file` para conferir os totais.

Fluxo:

```
START → planeja (write_todos) → dados (consultar_vendas, calcular) → escreve (write_file) → lê (read_file) → resposta final → END
```

Onde ficam os arquivos: o backend padrão guarda os arquivos no estado do grafo, não no disco. Por isso, eles aparecem em `estado["files"]`, e não na sua pasta.

Tudo fica registrado no MLflow, no experimento `deep-agent-arquivos`.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto, como nos outros notebooks.

O rastreamento usa o MLflow com um banco SQLite, `mlflow.db`, criado na pasta deste notebook. O `mlflow.langchain.autolog()` registra cada execução, com as chamadas ao modelo, ao plano, às ferramentas e aos arquivos.

In [1]:
import ast
import json
import operator
import os
import unicodedata

import mlflow
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from deepagents import create_deep_agent

load_dotenv(find_dotenv())

mlflow.set_tracking_uri("sqlite:///mlflow.db")
EXPERIMENTO = "deep-agent-arquivos"
experimento = mlflow.set_experiment(EXPERIMENTO)
mlflow.langchain.autolog()

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

2026/11/17 07:49:11 INFO mlflow.tracking.fluent: Experiment with name 'deep-agent-arquivos' does not exist. Creating a new experiment.


## 2. Ferramentas

As mesmas do notebook anterior:

- `consultar_vendas`: devolve as vendas de um mês. Os dados são simulados.
- `calcular`: faz contas com números e as quatro operações, usando `ast`. Nunca use `eval` com texto gerado por LLM.

As ferramentas de arquivo (`write_file`, `read_file` etc.) já vêm com o deep agent.

In [2]:
VENDAS_API = {
    "janeiro": {"cafe": 120, "pao": 300, "queijo": 45},
    "fevereiro": {"cafe": 135, "pao": 280, "queijo": 50},
    "marco": {"cafe": 150, "pao": 310, "queijo": 38},
}


@tool
def consultar_vendas(mes: str) -> str:
    """Consulta as vendas de um mês do primeiro trimestre. Retorna a quantidade vendida de cada produto em JSON. Meses válidos: janeiro, fevereiro, março."""
    chave = unicodedata.normalize("NFKD", mes.lower().strip()).encode("ascii", "ignore").decode()
    dados = VENDAS_API.get(chave)
    if dados is None:
        return json.dumps({"mes": mes, "erro": "mês não encontrado"}, ensure_ascii=False)
    return json.dumps({"mes": mes, "vendas": dados}, ensure_ascii=False)


OPERACOES = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}


def _avaliar(no: ast.AST) -> float:
    if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
        return no.value
    if isinstance(no, ast.BinOp) and type(no.op) in OPERACOES:
        return OPERACOES[type(no.op)](_avaliar(no.left), _avaliar(no.right))
    if isinstance(no, ast.UnaryOp) and isinstance(no.op, ast.USub):
        return -_avaliar(no.operand)
    raise ValueError("expressão não permitida")


@tool
def calcular(expressao: str) -> str:
    """Calcula uma expressão aritmética com números, + - * / e parênteses. Exemplo: '(120 + 135) * 2'."""
    return str(round(_avaliar(ast.parse(expressao, mode="eval").body), 2))


FERRAMENTAS = [consultar_vendas, calcular]

print(consultar_vendas.name, "|", calcular.name)

consultar_vendas | calcular


## 3. Criando o agente

O `TodoListMiddleware` adiciona o `write_todos`. O sistema de arquivos já vem no deep agent. O `system_prompt` pede a ordem: plano, dados, arquivo e, por fim, leitura do arquivo antes da resposta.

In [3]:
agente = create_deep_agent(
    model=llm,
    tools=FERRAMENTAS,
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "Você é um analista de vendas. Em tarefas com etapas, a primeira ação é chamar write_todos "
        "com um plano de 3 a 5 passos. Busque os dados com as ferramentas e faça as contas com calcular. "
        "Grave o relatório com write_file e, antes de responder, leia o arquivo com read_file. "
        "Responda em português, com os números usados."
    ),
)

## 4. Executando

A função `executar` roda a tarefa e mostra as mensagens na ordem em que aconteceram:

- `Plano` é a lista de passos da primeira chamada de `write_todos`. Chamadas seguintes atualizam o status.
- `→` é uma chamada de ferramenta, sua ou de arquivo.
- `←` é o resultado devolvido.
- `Resposta` é a mensagem final.

In [4]:
def resumir(texto: str, limite: int = 200) -> str:
    return texto if len(texto) <= limite else texto[:limite] + "..."


def executar(tarefa: str) -> dict:
    estado = agente.invoke({"messages": [HumanMessage(content=tarefa)]})

    linhas = [f"**Tarefa:** {tarefa}", ""]
    plano_mostrado = False
    for mensagem in estado["messages"][1:]:
        if isinstance(mensagem, AIMessage) and mensagem.tool_calls:
            for chamada in mensagem.tool_calls:
                if chamada["name"] == "write_todos" and not plano_mostrado:
                    linhas.append("**Plano:**")
                    for item in chamada["args"].get("todos", []):
                        linhas.append(f"- {item.get('content', item)}")
                    linhas.append("")
                    plano_mostrado = True
                elif chamada["name"] == "write_todos":
                    linhas.append("→ `write_todos` (atualiza o status dos passos)")
                else:
                    linhas.append(f"→ `{chamada['name']}({resumir(str(chamada['args']))})`")
        elif isinstance(mensagem, ToolMessage) and mensagem.name != "write_todos":
            linhas.append(f"← `{resumir(str(mensagem.content))}`")
        elif isinstance(mensagem, AIMessage) and mensagem.content:
            linhas.append("")
            linhas.append(f"**Resposta:** {mensagem.content}")
    display(Markdown("\n".join(linhas)))
    return estado


estado = executar(
    "Gere um relatório de vendas do primeiro trimestre com o total de cada produto e o mais vendido. "
    "Grave o relatório em /relatorio.md e leia o arquivo de volta para conferir os totais."
)

**Tarefa:** Gere um relatório de vendas do primeiro trimestre com o total de cada produto e o mais vendido. Grave o relatório em /relatorio.md e leia o arquivo de volta para conferir os totais.

**Plano:**
- Consultar vendas de janeiro, fevereiro e março
- Calcular totais por produto
- Escrever relatório em /relatorio.md
- Ler /relatorio.md para conferir

→ `consultar_vendas({'mes': 'janeiro'})`
← `{"mes": "janeiro", "vendas": {"cafe": 120, "pao": 300, "queijo": 45}}`
→ `consultar_vendas({'mes': 'fevereiro'})`
← `{"mes": "fevereiro", "vendas": {"cafe": 135, "pao": 280, "queijo": 50}}`
→ `consultar_vendas({'mes': 'março'})`
← `{"mes": "março", "vendas": {"cafe": 150, "pao": 310, "queijo": 38}}`
→ `write_todos` (atualiza o status dos passos)
→ `calcular({'expressao': '120 + 135 + 150'})`
← `405`
→ `calcular({'expressao': '300 + 280 + 310'})`
← `890`
→ `calcular({'expressao': '45 + 50 + 38'})`
← `133`
→ `write_todos` (atualiza o status dos passos)
→ `write_file({'file_path': '/relatorio.md', 'content': 'Relatório de Vendas - 1º Trimestre\n\nTotais por produto:\n- café: 405\n  (janeiro 120 + fevereiro 135 + março 150 = 405)\n- pão: 890\n  (janeiro 300 + fever...)`
← `Updated file /relatorio.md`
→ `write_todos` (atualiza o status dos passos)
→ `read_file({'file_path': '/relatorio.md'})`
← `@@ lines 1-11 of 11 @@
Relatório de Vendas - 1º Trimestre

Totais por produto:
- café: 405
  (janeiro 120 + fevereiro 135 + março 150 = 405)
- pão: 890
  (janeiro 300 + fevereiro 280 + março 310 = 890...`
→ `write_todos` (atualiza o status dos passos)

**Resposta:** Relatório gerado e conferido. Conteúdo de /relatorio.md lido:

Relatório de Vendas - 1º Trimestre

Totais por produto:
- café: 405
  (janeiro 120 + fevereiro 135 + março 150 = 405)
- pão: 890
  (janeiro 300 + fevereiro 280 + março 310 = 890)
- queijo: 133
  (janeiro 45 + fevereiro 50 + março 38 = 133)

Mais vendido: pão com 890 unidades.

## 5. Conferindo os arquivos

`estado["files"]` guarda os arquivos criados pelo agente, com o caminho como chave. Abaixo, a lista de arquivos e o conteúdo do relatório.

In [5]:
arquivos = estado.get("files") or {}
print("Arquivos criados:", list(arquivos.keys()))

conteudo = arquivos.get("/relatorio.md")
if conteudo is not None:
    texto = conteudo["content"] if isinstance(conteudo, dict) else conteudo
    texto = "\n".join(texto) if isinstance(texto, list) else str(texto)
    display(Markdown(texto))

Arquivos criados: ['/relatorio.md']


Relatório de Vendas - 1º Trimestre

Totais por produto:
- café: 405
  (janeiro 120 + fevereiro 135 + março 150 = 405)
- pão: 890
  (janeiro 300 + fevereiro 280 + março 310 = 890)
- queijo: 133
  (janeiro 45 + fevereiro 50 + março 38 = 133)

Mais vendido: pão com 890 unidades.


## 6. Conferindo a ordem

O esperado: `write_todos` primeiro, depois as ferramentas de dados, `write_file` para gravar e `read_file` para conferir.

In [6]:
ordem = []
for mensagem in estado["messages"]:
    if isinstance(mensagem, AIMessage):
        ordem.extend(chamada["name"] for chamada in mensagem.tool_calls)

print(" → ".join(ordem) if ordem else "nenhuma ferramenta chamada")
print("Planejou primeiro:", bool(ordem) and ordem[0] == "write_todos")
print("Gravou arquivo:", "write_file" in ordem)
print("Leu o arquivo de volta:", "read_file" in ordem)

write_todos → consultar_vendas → consultar_vendas → consultar_vendas → write_todos → calcular → calcular → calcular → write_todos → write_file → write_todos → read_file → write_todos
Planejou primeiro: True
Gravou arquivo: True
Leu o arquivo de volta: True


## 7. Acompanhando o rastreamento no MLflow

Cada execução virou um trace no experimento `deep-agent-arquivos`. Para ver:

1. Abra um terminal na pasta `notebooks/04-DeepAgents` (o caminho do `mlflow.db` é relativo a ela).
2. Rode:

```bash
uv run --project ../.. mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

3. Abra `http://127.0.0.1:5000` no navegador.
4. Em **Experiments**, escolha `deep-agent-arquivos` e abra a aba **Traces**.
5. Clique no trace mais recente. Os spans `TOOL` mostram `write_todos`, `consultar_vendas`, `calcular`, `write_file` e `read_file`, com entrada e saída de cada um. Nos spans de `write_file`, dá para ver o conteúdo gravado.
6. Pressione `Ctrl+C` no terminal para parar a interface.

A célula seguinte lista os spans pelo notebook, sem abrir a interface.

In [7]:
traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=10)
print(f"Traces registrados no experimento: {len(traces)}")

trace = mlflow.get_trace(traces.iloc[0]["trace_id"])
print(f"\nSpans do trace mais recente ({trace.info.trace_id}):")
for span in trace.data.spans:
    if span.span_type == "TOOL":
        print(f"- {span.name} ({span.span_type})")

Traces registrados no experimento: 1

Spans do trace mais recente (tr-e0856a365105b7e03916a83190400b7c):
- write_todos (TOOL)
- consultar_vendas (TOOL)
- consultar_vendas (TOOL)
- consultar_vendas (TOOL)
- write_todos (TOOL)
- calcular (TOOL)
- calcular (TOOL)
- calcular (TOOL)
- write_todos (TOOL)
- write_file (TOOL)
- write_todos (TOOL)
- read_file (TOOL)
- write_todos (TOOL)


/tmp/ipykernel_53905/2815719652.py:1: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=10)


## Observações

- O sistema de arquivos é um espaço de trabalho do agente. Com o backend padrão, os arquivos ficam no estado do grafo, não no disco. Para gravar em disco, é preciso configurar outro backend.
- O `read_file` depois do `write_file` é um passo de conferência. O `system_prompt` pede esse passo. Sem ele, o agente pode responder sem reler o arquivo.
- O resultado pode variar entre execuções. Compare o plano, as chamadas e os arquivos com a resposta final.